## Example 1: Setting up interactive matplotlib backend with ipympl

In [ ]:
import ipympl
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Example 1: Setting up interactive matplotlib backend with ipympl
print("Example 1: Setting up ipympl backend")

# Enable ipympl interactive backend
matplotlib.use('module://ipympl.backend_nbagg')

# Check backend configuration
current_backend = matplotlib.get_backend()
print(f"Current matplotlib backend: {current_backend}")
print(f"ipympl version: {getattr(ipympl, '__version__', 'unknown')}")
print(f"ipympl version info: {ipympl.version_info}")

# Create a simple interactive plot
x = np.linspace(0, 2*np.pi, 100)
y = np.sin(x)

fig, ax = plt.subplots(figsize=(8, 4))
line, = ax.plot(x, y, 'b-', linewidth=2)
ax.set_title('Interactive Sine Wave')
ax.set_xlabel('x')
ax.set_ylabel('sin(x)')
ax.grid(True)

# Check canvas type
print(f"Figure canvas type: {type(fig.canvas).__name__}")
print(f"Interactive toolbar available: {hasattr(fig.canvas, 'toolbar')}")

from ipympl.backend_nbagg import Canvas, Toolbar
import ipywidgets

assert current_backend == 'module://ipympl.backend_nbagg'
assert isinstance(ipympl.version_info, tuple) and len(ipympl.version_info) == 3
# With this backend every figure is drawn on an ipywidgets Canvas with an ipympl Toolbar
assert isinstance(fig.canvas, Canvas) and isinstance(fig.canvas, ipywidgets.DOMWidget)
assert isinstance(fig.canvas.toolbar, Toolbar)
assert fig.canvas.figure is fig
assert np.allclose(line.get_ydata(), np.sin(x))
# The canvas renders to an RGBA buffer of width x height = (8 x 4 inches) x dpi
fig.canvas.draw()
assert np.asarray(fig.canvas.buffer_rgba()).shape == (int(4 * fig.dpi), int(8 * fig.dpi), 4)

plt.show()
plt.close(fig)

## Example 2: Creating interactive plots with pan and zoom

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example 2: Creating interactive plots with pan and zoom
print("Example 2: Interactive plotting with pan and zoom")

# Create data for multiple plots
x = np.linspace(0, 10, 200)
data_sets = [
    {"y": np.sin(x), "label": "sin(x)", "color": "blue"},
    {"y": np.cos(x), "label": "cos(x)", "color": "red"}, 
    {"y": np.sin(x) * np.cos(x), "label": "sin(x)*cos(x)", "color": "green"}
]

# Create interactive figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle('Interactive Multi-Plot Dashboard')

# Plot different functions in subplots
for i, data in enumerate(data_sets[:3]):
    row, col = i // 2, i % 2
    ax = axes[row, col]
    ax.plot(x, data["y"], color=data["color"], label=data["label"])
    ax.set_title(f'Plot {i+1}: {data["label"]}')
    ax.grid(True, alpha=0.3)
    ax.legend()

# Create a scatter plot in the last subplot
np.random.seed(42)
scatter_x = np.random.randn(50)
scatter_y = np.random.randn(50)
axes[1, 1].scatter(scatter_x, scatter_y, alpha=0.6, c='purple')
axes[1, 1].set_title('Random Scatter Plot')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
print(f"Created {len(data_sets)+1} interactive plots")
print("Use pan/zoom tools in the toolbar to interact with plots")

assert [ax.get_title() for ax in axes.flat] == ['Plot 1: sin(x)', 'Plot 2: cos(x)', 'Plot 3: sin(x)*cos(x)', 'Random Scatter Plot']
assert axes[1, 1].collections[0].get_offsets().shape == (50, 2)

# The toolbar switches between pan and zoom modes, and Home restores the original view
toolbar = fig.canvas.toolbar
toolbar.pan()
assert toolbar.mode.name == 'PAN'
toolbar.zoom()
assert toolbar.mode.name == 'ZOOM'
toolbar.zoom()
assert toolbar.mode.name == 'NONE'
ax = axes[0, 0]
original_xlim = ax.get_xlim()
toolbar.push_current()
ax.set_xlim(2, 4)
toolbar.push_current()
assert ax.get_xlim() == (2, 4)
toolbar.home()
assert ax.get_xlim() == original_xlim

plt.show()
plt.close(fig)

## Example 3: Interactive figure with dynamic data updates

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example 3: Interactive figure with dynamic data updates
print("Example 3: Dynamic data visualization")

# Generate time series data
t = np.linspace(0, 4*np.pi, 200)
frequencies = [1, 2, 3]
amplitudes = [1, 0.5, 0.3]

# Create figure for animated-style updates
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 4*np.pi)
ax.set_ylim(-2, 2)
ax.set_title('Interactive Waveform Visualization')
ax.set_xlabel('Time')
ax.set_ylabel('Amplitude')
ax.grid(True, alpha=0.3)

# Plot multiple waveforms
lines = []
colors = ['blue', 'red', 'green']

for i, (freq, amp, color) in enumerate(zip(frequencies, amplitudes, colors)):
    y = amp * np.sin(freq * t)
    line, = ax.plot(t, y, color=color, linewidth=2, 
                   label=f'Wave {i+1}: {amp}*sin({freq}t)')
    lines.append(line)

# Add combined waveform
combined = sum(amp * np.sin(freq * t) for freq, amp in zip(frequencies, amplitudes))
combined_line, = ax.plot(t, combined, 'black', linewidth=3, 
                        linestyle='--', label='Combined')
lines.append(combined_line)

ax.legend()

print(f"Created {len(lines)} interactive waveforms")
print("Figure supports zooming, panning, and data point inspection")
print(f"Data points per waveform: {len(t)}")

assert len(lines) == 4 and len(ax.get_legend().get_texts()) == 4
assert np.allclose(combined_line.get_ydata(), 1 * np.sin(t) + 0.5 * np.sin(2 * t) + 0.3 * np.sin(3 * t))
# Updating a line's data in place is how interactive figures refresh
lines[0].set_ydata(np.cos(t))
assert np.allclose(lines[0].get_ydata(), np.cos(t))

plt.show()
plt.close(fig)

## Example 4: Interactive 3D plotting capabilities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example 4: Interactive 3D plotting capabilities
print("Example 4: Interactive 3D visualization")

# Create 3D data
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
U, V = np.meshgrid(u, v)

# Generate 3D surface data
X = 2 * np.cos(U) * np.sin(V)
Y = 2 * np.sin(U) * np.sin(V) 
Z = 2 * np.cos(V)

# Create 3D interactive plot
fig = plt.figure(figsize=(12, 5))

# Surface plot
ax1 = fig.add_subplot(121, projection='3d')
surface = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax1.set_title('Interactive 3D Surface')
ax1.set_xlabel('X axis')
ax1.set_ylabel('Y axis')
ax1.set_zlabel('Z axis')

# 3D scatter plot
ax2 = fig.add_subplot(122, projection='3d')
np.random.seed(42)
scatter_x = np.random.randn(100)
scatter_y = np.random.randn(100)
scatter_z = np.random.randn(100)
colors = np.random.rand(100)

scatter = ax2.scatter(scatter_x, scatter_y, scatter_z, c=colors, cmap='plasma')
ax2.set_title('Interactive 3D Scatter')
ax2.set_xlabel('X axis')
ax2.set_ylabel('Y axis')
ax2.set_zlabel('Z axis')

plt.colorbar(scatter, ax=ax2, shrink=0.5)

print("Created interactive 3D visualizations")
print("Use mouse to rotate, zoom, and pan in 3D space")
print(f"Surface mesh size: {X.shape}")
print(f"Scatter points: {len(scatter_x)}")

plt.tight_layout()

assert X.shape == Y.shape == Z.shape == (50, 50)
# Every surface point lies on a sphere of radius 2
assert np.allclose(X**2 + Y**2 + Z**2, 4.0)
assert ax1.name == '3d' and ax2.name == '3d'
assert ax1.get_zlabel() == 'Z axis'

plt.show()
plt.close(fig)

## Example 5: Interactive figure with toolbar and widget features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example 5: Interactive figure with toolbar and widget features
print("Example 5: Advanced interactive features")

# Create complex visualization with multiple interaction modes
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Interactive Dashboard with Toolbar Features', fontsize=14)

# Plot 1: Line plot with markers
x1 = np.linspace(0, 10, 50)
y1 = np.sin(x1) + 0.1 * np.random.randn(50)
ax1.plot(x1, y1, 'bo-', markersize=4)
ax1.set_title('Noisy Sine Wave')
ax1.grid(True)

# Plot 2: Heatmap
data = np.random.randn(20, 20)
im = ax2.imshow(data, cmap='RdBu', aspect='auto')
ax2.set_title('Random Heatmap')
plt.colorbar(im, ax=ax2)

# Plot 3: Bar chart
categories = ['A', 'B', 'C', 'D', 'E']
values = np.random.rand(5) * 100
bars = ax3.bar(categories, values, color=['red', 'green', 'blue', 'orange', 'purple'])
ax3.set_title('Interactive Bar Chart')
ax3.set_ylabel('Values')

# Plot 4: Contour plot  
x4 = np.linspace(-3, 3, 50)
y4 = np.linspace(-3, 3, 50)
X4, Y4 = np.meshgrid(x4, y4)
Z4 = np.exp(-(X4**2 + Y4**2))
contour = ax4.contour(X4, Y4, Z4, levels=10)
ax4.clabel(contour, inline=True, fontsize=8)
ax4.set_title('Contour Plot')

plt.tight_layout()

# List the tools the ipympl toolbar actually offers
tool_names = [item[0] for item in fig.canvas.toolbar.toolitems if item[0]]
print(f"Toolbar tools: {tool_names}")
assert {'Home', 'Pan', 'Zoom'} <= set(tool_names)

# Four subplots plus the heatmap's colorbar
print(f"Figure has {len(fig.axes)} axes")
assert len(fig.axes) == 5
assert len(bars) == 5 and [b.get_x() + b.get_width() / 2 for b in bars] == [0, 1, 2, 3, 4]
assert im.get_array().shape == (20, 20)
# The Gaussian peaks near 1 at the origin and falls towards 0 at the corners
assert 0.99 < Z4.max() <= 1.0 and Z4.min() < 1e-7

plt.show()
plt.close(fig)